In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r'data\customers.csv')

In [3]:
df

,customer_id,name,email,age,country,status
0,C001,Alice Johnson,alice@example.com,32,India,active
1,C002,Bob Smith,bob@example.com,45,India,active
2,C003,Charlie Brown,charlie@example.com,28,USA,active
3,C004,Diana Prince,diana@example.com,35,UK,inactive
4,C005,Ethan Hunt,ethan@example.com,41,USA,active
5,C006,Fiona Green,fiona@example.com,29,India,active
6,C007,George Miller,george@example.com,52,Canada,active
7,C008,Hannah Lee,hannah@example.com,24,India,active
8,C009,Ian Wright,ian@example.com,38,UK,active
9,C010,Julia Adams,julia@example.com,31,USA,inactive


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  10 non-null     str  
 1   name         10 non-null     str  
 2   email        10 non-null     str  
 3   age          10 non-null     int64
 4   country      10 non-null     str  
 5   status       10 non-null     str  
dtypes: int64(1), str(5)
memory usage: 612.0 bytes


In [5]:
#How many rows and columns does the dataset have?
print(df.shape) # 10 rows and 6 columns 

#Should customer_id be nullable? 
# no 
print(df.dtypes)

#Should customer_id be unique?
#yes 
print(df.isnull().sum())

#What values should status be allowed to contain?
#active and inactive 
print(df["customer_id"].nunique())

#What would be a reasonable validation rule for age?
#age should be between 10 and 100 


(10, 6)
customer_id      str
name             str
email            str
age            int64
country          str
status           str
dtype: object
customer_id    0
name           0
email          0
age            0
country        0
status         0
dtype: int64
10


In [6]:
print(df["country"].value_counts())
print(df["status"].value_counts())

country
India     4
USA       3
UK        2
Canada    1
Name: count, dtype: int64
status
active      8
inactive    2
Name: count, dtype: int64


### An Expectation

An **Expectation** is essentially a statement about what you expect your data to look like.

For example:

> `customer_id` should never be null.

or:

> `age` should be between 18 and 100.

or:

> `status` should only contain `active` or `inactive`.

In [7]:
#setting up expectations
import great_expectations as gx 

In [8]:
context = gx.get_context()

The idea is directionally right, but I'd refine it to:

> **The Data Context is the main entry point for managing GX configuration and metadata, including data sources, assets, expectation suites, validation workflows, and results.**

The actual **validation** is what evaluates Expectations against a Batch of data.

### Think of the responsibilities like this

**Data Context**

> "What does my GX project know about?"

**Data Source**

> "Where does my data come from?"

**Data Asset**

> "Which logical dataset am I validating?"

**Batch**

> "Which particular data am I validating right now?"

**Expectation**

> "What should be true about this data?"

**Validation**

> "Does this particular Batch satisfy those expectations?"

### Project 1 — GX Validation Flow

So eventually our Project 1 will look roughly like this:

```text
customers.csv
     ↓
Data Source
     ↓
Customer Data Asset
     ↓
Customer Batch
     ↓
Customer Expectations
     ├── customer_id not null
     ├── customer_id unique
     ├── age between 18 and 100
     ├── status in {active, inactive}
     └── email not null
     ↓
Validation
     ↓
Validation Results
     ↓
PASS / FAIL

In [9]:
print(type(context))

<class 'great_expectations.data_context.data_context.file_data_context.FileDataContext'>


EphemeralDataContext -> GX created the context temporarily in memory rather than loading a persistent GX project configuration from disk.

In [10]:
print(dir(context.data_sources))

['_DataSourceManager__crud_registry', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_bind_asset_factory_method_if_not_present', '_build_add_asset_docstring', '_data_context', '_datasource_passed_in', '_datasource_passed_in_as_only_argument', '_register_add_datasource', '_register_add_or_update_datasource', '_register_assets', '_register_crud_method', '_register_datasource', '_register_delete_datasource', '_register_update_datasource', '_validate_current_datasource_type', 'add_alloy', 'add_aurora', 'add_bigquery', 'add_citus', 'add_databricks_sql', 'add_fabric', 'add_fabric_powerbi', 'add_neon', 'add_

In [11]:
data_source = context.data_sources.add_pandas(name='customer_data_source')
print(data_source)

id: 7059582c-3bbc-49e3-9147-52d997a0ddc0
name: customer_data_source
type: pandas



### Why doesn't GX just take `df`?

This is a useful conceptual distinction.

The **Data Source** represents *where/how data is accessed*.

The actual DataFrame is the **data itself**.

For example, a Pandas Data Source says:

> "I'm going to work with data that GX accesses through pandas."

It doesn't yet say:

> "Validate this particular DataFrame."

That's what the next layers — **Data Asset, Batch Definition, and Batch** — will establish.

In [12]:

data_asset = data_source.add_dataframe_asset(name='customer')


In [13]:
print(data_asset)

batch_metadata: {}
id: fcae5632-0af7-43f8-9346-e887c8c64069
name: customer
type: dataframe



## The Next Concept: Batch Definition

We're now at an important question:

> **Which actual data should GX validate?**

A **Data Asset** tells GX:

> "This represents customer data."

But it doesn't yet identify the particular data we want to validate.

That's where the **Batch Definition** comes in.

Think about it this way:

```text
Data Source
    │
    │  "How do I access data?"
    ↓
Data Asset
    │
    │  "What logical dataset is this?"
    ↓
Batch Definition
    │
    │  "How do I identify/retrieve a particular batch?"
    ↓
Batch
    │
    │  "Here is the actual data"
    ↓
Validation

In [14]:
print(dir(data_asset))

['Config', '_EXCLUDE_FROM_READER_OPTIONS', '__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__concrete__', '__concrete_name__', '__config__', '__custom_root_type__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__exclude_fields__', '__fields__', '__fields_set__', '__firstlineno__', '__format__', '__ge__', '__get_validators__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__include_fields__', '__init__', '__init_subclass__', '__iter__', '__json_encoder__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameterized_bases__', '__parameters__', '__post_root_validators__', '__pre_root_validators__', '__pretty__', '__private_attributes__', '__reduce__', '__reduce_ex__', '__repr__', '__repr_args__', '__repr_name__', '__repr_str__', '__rich_repr__', '__schema_cache__', '__setattr__', '__setstate__', '__signature__', '__sizeof__', '__slots__', '__static_attributes__', '__str__', '__subclass

For our current Project 1, add_batch_definition_whole_dataframe is the better choice.

In [15]:
batch_definition  = data_asset.add_batch_definition_whole_dataframe(name='customers_batch')

In [16]:
print(batch_definition)

id=UUID('2bb5286c-2727-4a2d-99e6-6d7bb0c8ef7c') name='customers_batch' partitioner=None


In [17]:
print(dir(batch_definition))

['Config', '__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__concrete__', '__concrete_name__', '__config__', '__custom_root_type__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__exclude_fields__', '__fields__', '__fields_set__', '__firstlineno__', '__format__', '__ge__', '__get_validators__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__include_fields__', '__init__', '__init_subclass__', '__iter__', '__json_encoder__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameterized_bases__', '__parameters__', '__post_root_validators__', '__pre_root_validators__', '__pretty__', '__private_attributes__', '__reduce__', '__reduce_ex__', '__repr__', '__repr_args__', '__repr_name__', '__repr_str__', '__rich_repr__', '__schema_cache__', '__setattr__', '__setstate__', '__signature__', '__sizeof__', '__slots__', '__static_attributes__', '__str__', '__subclasshook__', '__try_update_forward_r

In [18]:
help(batch_definition.get_batch)

Help on method get_batch in module great_expectations.core.batch_definition:

get_batch(batch_parameters: 'Optional[BatchParameters]' = None) -> 'Batch' method of great_expectations.core.batch_definition.BatchDefinition instance
    --Public API--
    Retrieves a batch from the underlying asset. Defaults to the last batch
    from the asset's batch list.

    Args:
        batch_parameters: Additional parameters to be used in fetching the batch.
            Numeric batch parameters (e.g. year, month, day) take integer values;
            digit-string values are still accepted but are deprecated and emit a
            warning, with support for them removed in 2.0.

    Returns:
        A Batch of data.



In [19]:
batch = batch_definition.get_batch(    batch_parameters={
        "dataframe": df
    })

In [20]:
print(batch)

In [21]:
print(type(batch))

<class 'great_expectations.datasource.fluent.interfaces.Batch'>


In [22]:
print(dir(batch))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_batch_definition', '_batch_markers', '_batch_request', '_batch_spec', '_create_id', '_create_validator', '_data', '_data_asset', '_datasource', '_get_metrics_calculator', '_id', '_parse_metric_value', '_validate_expectation', '_validate_expectation_suite', 'batch_definition', 'batch_markers', 'batch_request', 'batch_spec', 'columns', 'compute_metrics', 'data', 'data_asset', 'datasource', 'head', 'id', 'metadata', 'validate']


In [23]:
print(help(batch))

Help on Batch in module great_expectations.datasource.fluent.interfaces object:

class Batch(builtins.object)
 |  Batch(
 |      datasource: 'Datasource',
 |      data_asset: 'DataAsset',
 |      batch_request: 'BatchRequest',
 |      data: 'BatchData',
 |      batch_markers: 'BatchMarkers',
 |      batch_spec: 'BatchSpec',
 |      batch_definition: 'LegacyBatchDefinition',
 |      metadata: 'Dict[str, Any] | None' = None
 |  )
 |
 |  --Public API--This represents a batch of data.
 |
 |  This is usually not the data itself but a hook to the data on an external datastore such as
 |  a spark or a sql database. An exception exists for pandas or any in-memory datastore.
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |      datasource: 'Datasource',
 |      data_asset: 'DataAsset',
 |      batch_request: 'BatchRequest',
 |      data: 'BatchData',
 |      batch_markers: 'BatchMarkers',
 |      batch_spec: 'BatchSpec',
 |      batch_definition: 'LegacyBatchDefinition',
 |      m

In [24]:
print(type(batch.data))

<class 'great_expectations.execution_engine.pandas_batch_data.PandasBatchData'>


In [25]:
batch.data.head()

""


## creating expectation

In [26]:

suite = context.suites.add(
    gx.ExpectationSuite(name='customer_quality_suite_v1')

)
print(suite)

DataContextError: Cannot add ExpectationSuite with name customer_quality_suite_v1 because it already exists.

## add expectations

In [ ]:
expectation  = gx.expectations.ExpectColumnValuesToNotBeNull(
    column = 'customer_id'
)

In [ ]:
expectation

ExpectColumnValuesToNotBeNull(id=None, meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='customer_id', mostly=1, row_condition=None, condition_parser=None)

In [ ]:
suite.add_expectation(expectation)

ExpectColumnValuesToNotBeNull(id='602d8e76-98a7-4b33-9c58-3b821e1ac62d', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='customer_id', mostly=1, row_condition=None, condition_parser=None)

In [ ]:
suite

{
  "name": "customer_quality_suite_v1",
  "id": "f00466fd-f13f-4c73-9ffc-2e58136e66fd",
  "expectations": [
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "602d8e76-98a7-4b33-9c58-3b821e1ac62d",
      "severity": "critical"
    }
  ],
  "meta": {
    "great_expectations_version": "1.23.1"
  },
  "notes": null
}

In [ ]:
unique_expectation = gx.expectations.ExpectColumnValuesToBeUnique(
    column = 'customer_id'
)

In [ ]:
suite.add_expectation(unique_expectation)

ExpectColumnValuesToBeUnique(id='cbf63011-d237-40f0-9607-614a456da495', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='customer_id', mostly=1, row_condition=None, condition_parser=None)

In [ ]:
suite

{
  "name": "customer_quality_suite_v1",
  "id": "f00466fd-f13f-4c73-9ffc-2e58136e66fd",
  "expectations": [
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "602d8e76-98a7-4b33-9c58-3b821e1ac62d",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_unique",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "cbf63011-d237-40f0-9607-614a456da495",
      "severity": "critical"
    }
  ],
  "meta": {
    "great_expectations_version": "1.23.1"
  },
  "notes": null
}

In [ ]:
print(dir(gx.expectations.ExpectColumnValuesToBeBetween))

['Config', '__abstractmethods__', '__annotations__', '__class__', '__class_vars__', '__config__', '__copy__', '__custom_root_type__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__exclude_fields__', '__fields__', '__fields_set__', '__firstlineno__', '__format__', '__ge__', '__get_validators__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__include_fields__', '__init__', '__init_subclass__', '__iter__', '__json_encoder__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__post_root_validators__', '__pre_root_validators__', '__pretty__', '__private_attributes__', '__reduce__', '__reduce_ex__', '__repr__', '__repr_args__', '__repr_name__', '__repr_str__', '__rich_repr__', '__schema_cache__', '__setattr__', '__setstate__', '__signature__', '__sizeof__', '__slots__', '__static_attributes__', '__str__', '__subclasshook__', '__try_update_forward_refs__', '__validators__', '_abc_impl', '_add_array_params', '_add_mostly_pct_param', '_atomic_diagnostic_obse

In [ ]:
help(gx.expectations.ExpectColumnValuesToBeBetween)

Help on class ExpectColumnValuesToBeBetween in module great_expectations.expectations.core.expect_column_values_to_be_between:

class ExpectColumnValuesToBeBetween(great_expectations.expectations.expectation.ColumnMapExpectation)
 |  ExpectColumnValuesToBeBetween(
 |      *,
 |      id: Optional[str] = None,
 |      meta: Optional[dict] = None,
 |      notes: Union[str, List[str], NoneType] = None,
 |      result_format: Union[great_expectations.core.result_format.ResultFormat, dict] = <ResultFormat.BASIC: 'BASIC'>,
 |      description: Optional[str] = None,
 |      catch_exceptions: bool = True,
 |      rendered_content: Optional[List[great_expectations.render.components.RenderedAtomicContent]] = None,
 |      severity: great_expectations.expectations.metadata_types.FailureSeverity = <FailureSeverity.CRITICAL: 'critical'>,
 |      windows: Optional[List[great_expectations.expectations.window.Window]] = None,
 |      batch_id: Optional[str] = None,
 |      column: pydantic.v1.schema.St

In [ ]:
#age between
age_between = gx.expectations.ExpectColumnValuesToBeBetween(
    column='age'
    ,min_value = 18
    ,max_value = 100

)

In [ ]:
suite.add_expectation(age_between)

ExpectColumnValuesToBeBetween(id='dfab93aa-7b3d-46a0-8d72-a62f6ae2cdf6', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='age', mostly=1, row_condition=None, condition_parser=None, min_value=18.0, max_value=100.0, strict_min=False, strict_max=False)

In [ ]:
print(suite)

{
  "name": "customer_quality_suite_v1",
  "id": "f00466fd-f13f-4c73-9ffc-2e58136e66fd",
  "expectations": [
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "602d8e76-98a7-4b33-9c58-3b821e1ac62d",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_unique",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "cbf63011-d237-40f0-9607-614a456da495",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_between",
      "kwargs": {
        "column": "age",
        "min_value": 18.0,
        "max_value": 100.0
      },
      "meta": {},
      "id": "dfab93aa-7b3d-46a0-8d72-a62f6ae2cdf6",
      "severity": "critical"
    }
  ],
  "meta": {
    "great_expectations_version": "1.23.1"
  },
  "notes": null
}


In [ ]:
help(gx.expectations.ExpectColumnDistinctValuesToBeInSet)

Help on class ExpectColumnDistinctValuesToBeInSet in module great_expectations.expectations.core.expect_column_distinct_values_to_be_in_set:

class ExpectColumnDistinctValuesToBeInSet(great_expectations.expectations.expectation.ColumnAggregateExpectation)
 |  ExpectColumnDistinctValuesToBeInSet(
 |      *,
 |      id: Optional[str] = None,
 |      meta: Optional[dict] = None,
 |      notes: Union[str, List[str], NoneType] = None,
 |      result_format: Union[great_expectations.core.result_format.ResultFormat, dict] = <ResultFormat.BASIC: 'BASIC'>,
 |      description: Optional[str] = None,
 |      catch_exceptions: bool = False,
 |      rendered_content: Optional[List[great_expectations.render.components.RenderedAtomicContent]] = None,
 |      severity: great_expectations.expectations.metadata_types.FailureSeverity = <FailureSeverity.CRITICAL: 'critical'>,
 |      windows: Optional[List[great_expectations.expectations.window.Window]] = None,
 |      batch_id: Optional[str] = None,
 |  

In [ ]:
status_allowed_values = gx.expectations.ExpectColumnDistinctValuesToBeInSet(
    column='status',
    value_set = ['active','inactive']
)

In [ ]:
suite.add_expectation(status_allowed_values)

ExpectColumnDistinctValuesToBeInSet(id='ae27598a-80b0-4062-a43c-aee3363fd43f', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=False, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='status', row_condition=None, condition_parser=None, value_set=['active', 'inactive'])

In [ ]:
suite

{
  "name": "customer_quality_suite_v1",
  "id": "f00466fd-f13f-4c73-9ffc-2e58136e66fd",
  "expectations": [
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "602d8e76-98a7-4b33-9c58-3b821e1ac62d",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_unique",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "cbf63011-d237-40f0-9607-614a456da495",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_between",
      "kwargs": {
        "column": "age",
        "min_value": 18.0,
        "max_value": 100.0
      },
      "meta": {},
      "id": "dfab93aa-7b3d-46a0-8d72-a62f6ae2cdf6",
      "severity": "critical"
    },
    {
      "type": "expect_column_distinct_values_to_be_in_set",
      "kwargs": {
        "column": "status",
        "value_set": [
          "active",
          "inactive"

In [ ]:
email_not_null = gx.expectations.ExpectColumnValuesToNotBeNull(
    column='email'

)
suite.add_expectation(email_not_null)
suite

{
  "name": "customer_quality_suite_v1",
  "id": "f00466fd-f13f-4c73-9ffc-2e58136e66fd",
  "expectations": [
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "602d8e76-98a7-4b33-9c58-3b821e1ac62d",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_unique",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "cbf63011-d237-40f0-9607-614a456da495",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_between",
      "kwargs": {
        "column": "age",
        "min_value": 18.0,
        "max_value": 100.0
      },
      "meta": {},
      "id": "dfab93aa-7b3d-46a0-8d72-a62f6ae2cdf6",
      "severity": "critical"
    },
    {
      "type": "expect_column_distinct_values_to_be_in_set",
      "kwargs": {
        "column": "status",
        "value_set": [
          "active",
          "inactive"

In [ ]:
help(batch.validate)

Help on method validate in module great_expectations.datasource.fluent.interfaces:

validate(
    expect: 'Expectation | ExpectationSuite',
    *,
    result_format: 'ResultFormatUnion' = <ResultFormat.SUMMARY: 'SUMMARY'>,
    expectation_parameters: 'Optional[SuiteParameterDict]' = None
) -> 'ExpectationValidationResult | ExpectationSuiteValidationResult' method of great_expectations.datasource.fluent.interfaces.Batch instance
    --Public API--
    Validate the Batch using the provided Expectation or Expectation Suite.

    Args:
        expect: The Expectation or Expectation Suite to validate.
        result_format: The format to return the validation results in.
        expectation_parameters: A dictionary of parameters values for any
                                expectations using parameterized values (the $PARAMETER syntax).
                                The keys are the parameter names and the values are the values
                                to be used for this validat

In [ ]:
validation_result = batch.validate(expect=suite)

Calculating Metrics: 100%|██████████| 27/27 [00:00<00:00, 981.09it/s] 


In [ ]:
print(validation_result)

{
  "success": true,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "customer_data_source-customer",
          "column": "customer_id"
        },
        "meta": {},
        "id": "602d8e76-98a7-4b33-9c58-3b821e1ac62d",
        "severity": "critical"
      },
      "result": {
        "element_count": 10,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_be_unique",
        "kwargs": {
          "batch_id": "customer_data_source-customer",
          "column"

In [ ]:
for expectation in suite.expectations:
    print(expectation)

id='602d8e76-98a7-4b33-9c58-3b821e1ac62d' meta=None notes=None result_format=<ResultFormat.BASIC: 'BASIC'> description=None catch_exceptions=True rendered_content=None severity=<FailureSeverity.CRITICAL: 'critical'> windows=None batch_id=None column='customer_id' mostly=1 row_condition=None condition_parser=None
id='cbf63011-d237-40f0-9607-614a456da495' meta=None notes=None result_format=<ResultFormat.BASIC: 'BASIC'> description=None catch_exceptions=True rendered_content=None severity=<FailureSeverity.CRITICAL: 'critical'> windows=None batch_id=None column='customer_id' mostly=1 row_condition=None condition_parser=None
id='dfab93aa-7b3d-46a0-8d72-a62f6ae2cdf6' meta=None notes=None result_format=<ResultFormat.BASIC: 'BASIC'> description=None catch_exceptions=True rendered_content=None severity=<FailureSeverity.CRITICAL: 'critical'> windows=None batch_id=None column='age' mostly=1 row_condition=None condition_parser=None min_value=18.0 max_value=100.0 strict_min=False strict_max=False
i

In [ ]:
[x for x in dir(suite) if "expectation" in x.lower()]

['_add_expectation',
 '_build_expectation',
 '_expectations_are_equalish',
 '_find_expectation_indexes',
 '_process_expectation',
 '_save_expectation',
 'add_expectation',
 'add_expectation_configuration',
 'add_expectation_configurations',
 'delete_expectation',
 'expectation_configurations',
 'expectations',
 'remove_expectation']

In [ ]:
help(suite.delete_expectation)

Help on method delete_expectation in module great_expectations.core.expectation_suite:

delete_expectation(expectation: 'Expectation') -> 'Expectation' method of great_expectations.core.expectation_suite.ExpectationSuite instance
    --Public API--Delete an Expectation from the collection.

    Example:
        >>> suite.delete_expectation(suite.expectations[0])

    Raises:
        KeyError: Expectation not found in suite.



In [ ]:
for i, expectation in enumerate(suite.expectations):
    print(i, expectation)

0 id='602d8e76-98a7-4b33-9c58-3b821e1ac62d' meta=None notes=None result_format=<ResultFormat.BASIC: 'BASIC'> description=None catch_exceptions=True rendered_content=None severity=<FailureSeverity.CRITICAL: 'critical'> windows=None batch_id=None column='customer_id' mostly=1 row_condition=None condition_parser=None
1 id='cbf63011-d237-40f0-9607-614a456da495' meta=None notes=None result_format=<ResultFormat.BASIC: 'BASIC'> description=None catch_exceptions=True rendered_content=None severity=<FailureSeverity.CRITICAL: 'critical'> windows=None batch_id=None column='customer_id' mostly=1 row_condition=None condition_parser=None
2 id='dfab93aa-7b3d-46a0-8d72-a62f6ae2cdf6' meta=None notes=None result_format=<ResultFormat.BASIC: 'BASIC'> description=None catch_exceptions=True rendered_content=None severity=<FailureSeverity.CRITICAL: 'critical'> windows=None batch_id=None column='age' mostly=1 row_condition=None condition_parser=None min_value=18.0 max_value=100.0 strict_min=False strict_max=F

In [ ]:
print(len(suite.expectations))

5


In [ ]:
validation_result = batch.validate(
    expect=suite
)

Calculating Metrics: 100%|██████████| 27/27 [00:00<00:00, 1091.53it/s]


In [ ]:
print(validation_result)

{
  "success": true,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "customer_data_source-customer",
          "column": "customer_id"
        },
        "meta": {},
        "id": "602d8e76-98a7-4b33-9c58-3b821e1ac62d",
        "severity": "critical"
      },
      "result": {
        "element_count": 10,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_be_unique",
        "kwargs": {
          "batch_id": "customer_data_source-customer",
          "column"

to interpret wrong data

In [ ]:
bad_df = df.copy()

In [ ]:
bad_df.loc[1, "customer_id"] = bad_df.loc[0, "customer_id"]

In [ ]:
bad_df

,customer_id,name,email,age,country,status
0,C001,Alice Johnson,alice@example.com,32,India,active
1,C001,Bob Smith,bob@example.com,45,India,active
2,C003,Charlie Brown,charlie@example.com,28,USA,active
3,C004,Diana Prince,diana@example.com,35,UK,inactive
4,C005,Ethan Hunt,ethan@example.com,41,USA,active
5,C006,Fiona Green,fiona@example.com,29,India,active
6,C007,George Miller,george@example.com,52,Canada,active
7,C008,Hannah Lee,hannah@example.com,24,India,active
8,C009,Ian Wright,ian@example.com,38,UK,active
9,C010,Julia Adams,julia@example.com,31,USA,inactive


In [ ]:
bad_batch = batch_definition.get_batch(
    batch_parameters={
        "dataframe": bad_df
    }
)

In [ ]:
bad_result = bad_batch.validate(
    expect=suite
)

Calculating Metrics: 100%|██████████| 27/27 [00:00<00:00, 879.09it/s]


In [ ]:
print(bad_result)

{
  "success": false,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "customer_data_source-customer",
          "column": "customer_id"
        },
        "meta": {},
        "id": "602d8e76-98a7-4b33-9c58-3b821e1ac62d",
        "severity": "critical"
      },
      "result": {
        "element_count": 10,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": false,
      "expectation_config": {
        "type": "expect_column_values_to_be_unique",
        "kwargs": {
          "batch_id": "customer_data_source-customer",
          "colum

In [ ]:
bad_df2 = df.copy()
bad_df2.loc[1, "customer_id"] = bad_df2.loc[0, "customer_id"]
bad_df2.loc[2, "status"] = "pending"


In [ ]:
bad_batch2 = batch_definition.get_batch(
    batch_parameters={
        "dataframe": bad_df2
    }
)

In [ ]:
bad_result2 = bad_batch2.validate(
    expect=suite
)

Calculating Metrics: 100%|██████████| 27/27 [00:00<00:00, 941.84it/s] 


In [ ]:
print(bad_result2)

{
  "success": false,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "customer_data_source-customer",
          "column": "customer_id"
        },
        "meta": {},
        "id": "602d8e76-98a7-4b33-9c58-3b821e1ac62d",
        "severity": "critical"
      },
      "result": {
        "element_count": 10,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": false,
      "expectation_config": {
        "type": "expect_column_values_to_be_unique",
        "kwargs": {
          "batch_id": "customer_data_source-customer",
          "colum

In [ ]:
#email 
[x for x in dir(gx.expectations) if "Match" in x or "Regex" in x]

['ExpectColumnValuesToMatchJsonSchema',
 'ExpectColumnValuesToMatchLikePattern',
 'ExpectColumnValuesToMatchLikePatternList',
 'ExpectColumnValuesToMatchRegex',
 'ExpectColumnValuesToMatchRegexList',
 'ExpectColumnValuesToMatchStrftimeFormat',
 'ExpectColumnValuesToNotMatchLikePattern',
 'ExpectColumnValuesToNotMatchLikePatternList',
 'ExpectColumnValuesToNotMatchRegex',
 'ExpectColumnValuesToNotMatchRegexList',
 'ExpectQueryResultsToMatchComparison',
 'ExpectTableColumnsToMatchOrderedList',
 'ExpectTableColumnsToMatchSet']

In [ ]:
help(gx.expectations.ExpectColumnValuesToMatchRegex)

Help on class ExpectColumnValuesToMatchRegex in module great_expectations.expectations.core.expect_column_values_to_match_regex:

class ExpectColumnValuesToMatchRegex(great_expectations.expectations.expectation.ColumnMapExpectation)
 |  ExpectColumnValuesToMatchRegex(
 |      *,
 |      id: Optional[str] = None,
 |      meta: Optional[dict] = None,
 |      notes: Union[str, List[str], NoneType] = None,
 |      result_format: Union[great_expectations.core.result_format.ResultFormat, dict] = <ResultFormat.BASIC: 'BASIC'>,
 |      description: Optional[str] = None,
 |      catch_exceptions: bool = True,
 |      rendered_content: Optional[List[great_expectations.render.components.RenderedAtomicContent]] = None,
 |      severity: great_expectations.expectations.metadata_types.FailureSeverity = <FailureSeverity.CRITICAL: 'critical'>,
 |      windows: Optional[List[great_expectations.expectations.window.Window]] = None,
 |      batch_id: Optional[str] = None,
 |      column: pydantic.v1.schem

In [ ]:
email_regex = gx.expectations.ExpectColumnValuesToMatchRegex(
    column = 'email',
    regex = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

)

In [ ]:
print(email_regex)

id=None meta=None notes=None result_format=<ResultFormat.BASIC: 'BASIC'> description=None catch_exceptions=True rendered_content=None severity=<FailureSeverity.CRITICAL: 'critical'> windows=None batch_id=None column='email' mostly=1 row_condition=None condition_parser=None regex='^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$'


In [ ]:
suite.add_expectation(email_regex)

ExpectColumnValuesToMatchRegex(id='cc80d875-772e-4d1f-b209-f026c6a3b85f', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='email', mostly=1, row_condition=None, condition_parser=None, regex='^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$')

In [ ]:
email_batch = batch_definition.get_batch(
    batch_parameters= {
        'dataframe': bad_df2
    }

)

In [ ]:
results = email_batch.validate(
    expect = suite
)

Calculating Metrics: 100%|██████████| 32/32 [00:00<00:00, 1052.00it/s]


In [ ]:
print(results)

{
  "success": false,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "customer_data_source-customer",
          "column": "customer_id"
        },
        "meta": {},
        "id": "602d8e76-98a7-4b33-9c58-3b821e1ac62d",
        "severity": "critical"
      },
      "result": {
        "element_count": 10,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": false,
      "expectation_config": {
        "type": "expect_column_values_to_be_unique",
        "kwargs": {
          "batch_id": "customer_data_source-customer",
          "colum

In [ ]:
bad_email_df = df.copy()

In [ ]:
bad_email_df.loc[3,'email'] = 'not-an-email'

In [ ]:
bad_email_batch = batch_definition.get_batch(
    batch_parameters = {
        'dataframe':bad_email_df
    }
)

In [ ]:
bad_email_results = bad_email_batch.validate(
    expect = suite
)

Calculating Metrics: 100%|██████████| 32/32 [00:00<00:00, 883.36it/s]


In [ ]:
print(bad_email_results)

{
  "success": false,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "customer_data_source-customer",
          "column": "customer_id"
        },
        "meta": {},
        "id": "602d8e76-98a7-4b33-9c58-3b821e1ac62d",
        "severity": "critical"
      },
      "result": {
        "element_count": 10,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_be_unique",
        "kwargs": {
          "batch_id": "customer_data_source-customer",
          "column

saving the suite

In [ ]:
print(context.suites.all())

[{
  "name": "customer_quality_suite_v1",
  "id": "f00466fd-f13f-4c73-9ffc-2e58136e66fd",
  "expectations": [
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "602d8e76-98a7-4b33-9c58-3b821e1ac62d",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_unique",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "cbf63011-d237-40f0-9607-614a456da495",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_between",
      "kwargs": {
        "column": "age",
        "min_value": 18.0,
        "max_value": 100.0
      },
      "meta": {},
      "id": "dfab93aa-7b3d-46a0-8d72-a62f6ae2cdf6",
      "severity": "critical"
    },
    {
      "type": "expect_column_distinct_values_to_be_in_set",
      "kwargs": {
        "column": "status",
        "value_set": [
          "active",
          "inactive

In [ ]:
print(context.data_sources.all())

{'customer_data_source': PandasDatasource(type='pandas', name='customer_data_source', id=UUID('248e7d78-3d01-473d-8dee-00075df507c6'), assets=[DataFrameAsset(name='customer', type='dataframe', id=UUID('f5da1dad-07f1-421b-b811-bd8ac68cb895'), order_by=[], batch_metadata={}, batch_definitions=[BatchDefinition(id=UUID('91601f81-3fa2-4c32-ba56-15b16e81b925'), name='customers_batch', partitioner=None)])])}


In [ ]:
print(context)

{
  "checkpoint_store_name": "checkpoint_store",
  "config_version": 4,
  "data_docs_sites": {
    "local_site": {
      "class_name": "SiteBuilder",
      "show_how_to_buttons": true,
      "store_backend": {
        "class_name": "TupleFilesystemStoreBackend",
        "base_directory": "C:\\Users\\Thiru\\AppData\\Local\\Temp\\tmp0ax_po9f"
      },
      "site_index_builder": {
        "class_name": "DefaultSiteIndexBuilder"
      }
    }
  },
  "expectations_store_name": "expectations_store",
  "fluent_datasources": {
    "customer_data_source": {
      "type": "pandas",
      "name": "customer_data_source",
      "id": "248e7d78-3d01-473d-8dee-00075df507c6",
      "assets": [
        {
          "name": "customer",
          "type": "dataframe",
          "id": "f5da1dad-07f1-421b-b811-bd8ac68cb895",
          "batch_metadata": {},
          "batch_definitions": [
            {
              "id": "91601f81-3fa2-4c32-ba56-15b16e81b925",
              "name": "customers_batch"
      

Data docs

In [ ]:
print([x for x in dir(context) if "doc" in x.lower()])

['__doc__', '_build_data_docs', '_clean_data_docs_site', '_init_site_builder_for_data_docs_site_creation', '_open_data_docs', 'add_data_docs_site', 'build_data_docs', 'clean_data_docs', 'delete_data_docs_site', 'get_docs_sites_urls', 'list_data_docs_sites', 'open_data_docs', 'update_data_docs_site']


In [ ]:
help(context.build_data_docs)

Help on method build_data_docs in module great_expectations.data_context.data_context.abstract_data_context:

build_data_docs(
    site_names: 'list[str] | None' = None,
    resource_identifiers: 'list[ExpectationSuiteIdentifier] | list[ValidationResultIdentifier] | None' = None,
    dry_run: 'bool' = False,
    build_index: 'bool' = True
) -> 'dict[str, str]' method of great_expectations.data_context.data_context.ephemeral_data_context.EphemeralDataContext instance
    --Public API--Build Data Docs for your project.

    --Documentation--
        - https://docs.greatexpectations.io/docs/terms/data_docs/

    Args:
        site_names: if specified, build data docs only for these sites, otherwise,
            build all the sites specified in the context's config
        resource_identifiers: a list of resource identifiers (ExpectationSuiteIdentifier,
            ValidationResultIdentifier). If specified, rebuild HTML
            (or other views the data docs sites are rendering) only fo

In [ ]:
data_docs = context.build_data_docs()
print(data_docs)

{'local_site': 'file://C:\\Users\\Thiru\\AppData\\Local\\Temp\\tmp0ax_po9f\\index.html'}


In [ ]:
print(data_docs["local_site"])

file://C:\Users\Thiru\AppData\Local\Temp\tmp0ax_po9f\index.html


In [ ]:
print(data_docs)

{'local_site': 'file://C:\\Users\\Thiru\\AppData\\Local\\Temp\\tmp0ax_po9f\\index.html'}


In [ ]:
print(type(context))

<class 'great_expectations.data_context.data_context.ephemeral_data_context.EphemeralDataContext'>


In [ ]:
[x for x in dir(context) if "project" in x.lower() or "context" in x.lower()]

['_construct_data_context_id',
 '_data_context_id',
 '_init_project_config',
 '_project_config',
 '_save_project_config',
 'convert_to_file_context',
 'data_context_id',
 'get_or_create_data_context_config',
 'project_config_with_variables_substituted',
 'update_project_config']

In [ ]:
help(gx.get_context)

Help on function get_context in module great_expectations.data_context.data_context.context_factory:

get_context(
    project_config: 'DataContextConfig | Mapping | None' = None,
    context_root_dir: 'PathStr | None' = None,
    project_root_dir: 'PathStr | None' = None,
    runtime_environment: 'dict | None' = None,
    cloud_base_url: 'str | None' = None,
    cloud_access_token: 'str | None' = None,
    cloud_organization_id: 'str | None' = None,
    cloud_workspace_id: 'str | None' = None,
    cloud_mode: 'bool | None' = None,
    user_agent_str: 'str | None' = None,
    mode: 'ContextModes | None' = None
) -> 'AbstractDataContext'
    --Public API--Method to return the appropriate Data Context depending on parameters and environment.

    Usage:
        `import great_expectations as gx`

        `my_context = gx.get_context(<insert_your_parameters>)`

    This method returns the appropriate Data Context based on which parameters you've passed and / or your environment configurati

In [ ]:
print(type(context))

<class 'great_expectations.data_context.data_context.ephemeral_data_context.EphemeralDataContext'>


In [ ]:
from pathlib import Path

print(Path.cwd())

d:\Study\Data Engineering\great_learning\great-expectations-learning\projects\01-customer-data-quality


In [1]:
import great_expectations as gx

context = gx.get_context(
    project_root_dir="."
)

print(type(context))

<class 'great_expectations.data_context.data_context.file_data_context.FileDataContext'>


In [2]:
from pathlib import Path

for path in Path(".").iterdir():
    print(path)

data
gx
inspect_data.ipynb
README.md


In [3]:
print(context.root_directory)

d:\Study\Data Engineering\great_learning\great-expectations-learning\projects\01-customer-data-quality\gx


In [4]:
from pathlib import Path

gx_dir = Path("gx")

for path in gx_dir.rglob("*"):
    print(path)

gx\.gitignore
gx\checkpoints
gx\expectations
gx\great_expectations.yml
gx\plugins
gx\uncommitted
gx\validation_definitions
gx\expectations\.ge_store_backend_id
gx\expectations\customer_quality_suite_v1.json
gx\plugins\custom_data_docs
gx\uncommitted\config_variables.yml
gx\uncommitted\data_docs
gx\uncommitted\validations
gx\uncommitted\validations\.ge_store_backend_id
gx\plugins\custom_data_docs\renderers
gx\plugins\custom_data_docs\styles
gx\plugins\custom_data_docs\views
gx\plugins\custom_data_docs\styles\data_docs_custom_styles.css


In [5]:
from pathlib import Path

config_file = Path("gx/great_expectations.yml")

print(config_file.read_text())

# Welcome to Great Expectations! Always know what to expect from your data.
#
# Here you can define datasources, batch kwargs generators, integrations and
# more. This file is intended to be committed to your repo. For help with
# configuration please:
#   - Read our docs: https://docs.greatexpectations.io/docs/guides/connecting_to_your_data/connect_to_data_overview/#2-configure-your-datasource
#   - Join our slack channel: http://greatexpectations.io/slack

# config_version refers to the syntactic version of this config file, and is used in maintaining backwards compatibility
# It is auto-generated and usually does not need to be changed.
config_version: 4.0

# This config file supports variable substitution which enables: 1) keeping
# secrets out of source control & 2) environment-based configuration changes
# such as staging vs prod.
#
# When GX encounters substitution syntax (like `my_key: ${my_value}` or
# `my_key: $my_value`) in the great_expectations.yml file, it will attempt
# 

In [6]:
print(context.suites.all())

[{
  "name": "customer_quality_suite_v1",
  "id": "7345e587-1a34-4690-a9d1-9bd953fdb693",
  "expectations": [
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "484b768d-65ab-49fd-b8a7-e60a6b750825",
      "severity": "critical"
    }
  ],
  "meta": {
    "great_expectations_version": "1.23.1"
  },
  "notes": null
}]


In [7]:
print(context.suites)

In [19]:
[x for x in dir(context) if "del" in dir(context.suites)]

[]

In [8]:
context.suites.all()

[{
   "name": "customer_quality_suite_v1",
   "id": "7345e587-1a34-4690-a9d1-9bd953fdb693",
   "expectations": [
     {
       "type": "expect_column_values_to_not_be_null",
       "kwargs": {
         "column": "customer_id"
       },
       "meta": {},
       "id": "484b768d-65ab-49fd-b8a7-e60a6b750825",
       "severity": "critical"
     }
   ],
   "meta": {
     "great_expectations_version": "1.23.1"
   },
   "notes": null
 }]

In [10]:
suite = context.suites.add(
    gx.ExpectationSuite(
        name="customer_quality_suite_v2"
    )
)

In [11]:
print(context.suites.all())

[{
  "name": "customer_quality_suite_v1",
  "id": "7345e587-1a34-4690-a9d1-9bd953fdb693",
  "expectations": [
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "484b768d-65ab-49fd-b8a7-e60a6b750825",
      "severity": "critical"
    }
  ],
  "meta": {
    "great_expectations_version": "1.23.1"
  },
  "notes": null
}, {
  "name": "customer_quality_suite_v2",
  "id": "58206d32-bf95-472d-bad7-20718c6ea9ba",
  "expectations": [],
  "meta": {
    "great_expectations_version": "1.23.1"
  },
  "notes": null
}]


In [20]:
dir(suite)

['Config',
 '__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_vars__',
 '__config__',
 '__copy__',
 '__custom_root_type__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__exclude_fields__',
 '__fields__',
 '__fields_set__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__get_validators__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__include_fields__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__json_encoder__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__post_root_validators__',
 '__pre_root_validators__',
 '__pretty__',
 '__private_attributes__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__repr_args__',
 '__repr_name__',
 '__repr_str__',
 '__rich_repr__',
 '__schema_cache__',
 '__setattr__',
 '__setstate__',
 '__signature__',
 '__sizeof__',
 '__slots__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__try_update_forward_refs__',
 '__validators__',
 '_abc_impl',
 '_add_array_p

In [12]:
expectation = gx.expectations.ExpectColumnValuesToNotBeNull(
    column="customer_id"
)

suite = suite.add_expectation(expectation)

In [ ]:
print(suite.expectations)

AttributeError: 'ExpectColumnValuesToNotBeNull' object has no attribute 'expectations'

In [ ]:
context.suites.all()

[{
   "name": "customer_quality_suite_v1",
   "id": "7345e587-1a34-4690-a9d1-9bd953fdb693",
   "expectations": [
     {
       "type": "expect_column_values_to_not_be_null",
       "kwargs": {
         "column": "customer_id"
       },
       "meta": {},
       "id": "484b768d-65ab-49fd-b8a7-e60a6b750825",
       "severity": "critical"
     }
   ],
   "meta": {
     "great_expectations_version": "1.23.1"
   },
   "notes": null
 }]

In [ ]:
context.suites.get("customer_quality_suite_v1")

{
  "name": "customer_quality_suite_v1",
  "id": "7345e587-1a34-4690-a9d1-9bd953fdb693",
  "expectations": [
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "customer_id"
      },
      "meta": {},
      "id": "484b768d-65ab-49fd-b8a7-e60a6b750825",
      "severity": "critical"
    }
  ],
  "meta": {
    "great_expectations_version": "1.23.1"
  },
  "notes": null
}

In [ ]:
suite.expectations

AttributeError: 'ExpectColumnValuesToNotBeNull' object has no attribute 'expectations'

In [ ]:
#customer id should be unique 

expectation = gx.expectations.ExpectColumnValuesToBeUnique(
    column="customer_id"
)

In [ ]:
suite.add_expectation(expectation)

AttributeError: 'ExpectColumnValuesToNotBeNull' object has no attribute 'add_expectation'